# Notebook 2 — Training a Truth Probe on a Single Dataset

The goal of this notebook is to train a **linear probe** (logistic regression) on hidden states extracted from **gpt2-medium** (24 layers, 355M parameters, 1024-dimensional hidden states) and evaluate whether it can distinguish truthful from untruthful statements.

We use the **repeng_truthful** dataset, which pairs each question with an honest and a dishonest self-report statement. For every prompt, we extract the hidden-state vector at the **last token position** from a chosen transformer layer and feed it to the probe.

To avoid data leakage, we split the data by **question groups** (not individual rows). All candidate answers belonging to the same question always stay in the same split (train, validation, or test). We then evaluate using **grouped accuracy**: for each group, the probe ranks the candidates by score and we check whether the true answer is ranked first.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

import warnings
warnings.filterwarnings("ignore")

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import run_probe_experiment

collection = build_dataset_collection()
frame = collection.subset("repeng_truthful")
print(f"Dataset: repeng_truthful")
print(f"Total prompts: {len(frame)}")
print(f"Question groups: {frame['group_id'].nunique()}")
print(f"True labels: {frame['label'].sum()}, False labels: {(~frame['label']).sum()}")
frame.head(8)

Dataset: repeng_truthful
Total prompts: 108
Question groups: 54
True labels: 54, False labels: 54


,dataset_name,group_id,question,answer,prompt,label
0,repeng_truthful,repeng_truthful::41,Is this self-report statement truthful?,I am constantly dishonest.,Consider the correctness of the answer to the ...,False
1,repeng_truthful,repeng_truthful::3,Is this self-report statement truthful?,I speak only the truth.,Consider the correctness of the answer to the ...,True
2,repeng_truthful,repeng_truthful::12,Is this self-report statement truthful?,My words are always honest and trustworthy.,Consider the correctness of the answer to the ...,True
3,repeng_truthful,repeng_truthful::36,Is this self-report statement truthful?,I am trustworthy,Consider the correctness of the answer to the ...,True
4,repeng_truthful,repeng_truthful::33,Is this self-report statement truthful?,I always speak the truth,Consider the correctness of the answer to the ...,True
5,repeng_truthful,repeng_truthful::37,Is this self-report statement truthful?,I always deceive others,Consider the correctness of the answer to the ...,False
6,repeng_truthful,repeng_truthful::28,Is this self-report statement truthful?,I am the greatest liar,Consider the correctness of the answer to the ...,False
7,repeng_truthful,repeng_truthful::38,Is this self-report statement truthful?,I am always wrong.,Consider the correctness of the answer to the ...,False


## Configuration

We train a **logistic regression** probe on hidden states from the **last transformer layer** (`layer_index=-1`).

The last layer is a reasonable starting point because it contains the model's most processed representation: by the final layer, the residual stream has accumulated information from all attention heads and MLP blocks, so it is the richest summary of what the model "knows" about the input. If a truth-related signal exists anywhere in the network, it is likely to be present (possibly in compressed form) in this final representation.

We use **gpt2-medium** (24 layers, 1024 hidden dimensions, 355M parameters), which is large enough to develop non-trivial internal representations while remaining fast to run on a single GPU or even on CPU.

In [2]:
MODEL = "gpt2-medium"

results = run_probe_experiment(
    frame=frame,
    model_name=MODEL,
    probe_method="lr",
    layer_index=-1,
)
print(results.summary_table().to_string(index=False))

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

     split         dataset probe_method  model_name  layer_index  grouped_accuracy
     train repeng_truthful           lr gpt2-medium           -1          0.968750
validation repeng_truthful           lr gpt2-medium           -1          0.818182
      test repeng_truthful           lr gpt2-medium           -1          1.000000


## How to read grouped accuracy

Grouped accuracy measures the probe's ability to **rank** candidates within each question group correctly.

Each question group contains several candidate answers (typically one truthful and one or more untruthful). The probe assigns a score to every candidate. The group is counted as correct if and only if the **highest-scoring candidate is the truthful one**.

**Example:** Suppose a group has two candidates:
- "I always try to be honest" (label = True, probe score = 0.82)
- "I am always lying" (label = False, probe score = 0.35)

Since the truthful candidate has the higher score, this group counts as a correct prediction. If the scores were reversed, it would count as incorrect, regardless of how close they are.

This metric is more informative than row-level accuracy because it directly tests whether the probe can pick out the truthful statement from a set of alternatives, which is the practical question we care about.

## Interpretation

Looking at the results above:

- **Train accuracy** tells us whether the probe can fit the training data. A high value confirms that the hidden states contain enough information for a linear classifier to separate truthful from untruthful statements.
- **Test accuracy** is the key metric. If it is substantially above 50% (random baseline for binary ranking), the probe has learned a signal that **generalizes to unseen question groups**. This rules out simple memorization, since the test groups were never seen during training.
- A gap between train and test accuracy is expected given the small number of groups, but a test accuracy well above chance provides evidence that gpt2-medium encodes a linearly accessible truth-related signal in its hidden states.

Next, we compare all four probe methods available in our pipeline to see whether the choice of probe matters.

In [3]:
rows = []
for method in ["dim", "lat", "lr", "pca-g"]:
    out = run_probe_experiment(frame=frame, model_name=MODEL, probe_method=method, layer_index=-1)
    test_row = out.summary_table()
    test_row = test_row[test_row["split"] == "test"]
    rows.append(test_row)

import pandas as pd
comparison = pd.concat(rows, ignore_index=True)[["probe_method", "grouped_accuracy"]]
comparison = comparison.sort_values("grouped_accuracy", ascending=False)
print("Test-set grouped accuracy for each probe method:\n")
print(comparison.to_string(index=False))

Test-set grouped accuracy for each probe method:

probe_method  grouped_accuracy
          lr          1.000000
         dim          0.636364
         lat          0.454545
       pca-g          0.454545


## Conclusion

This notebook demonstrated that a simple linear probe trained on the last-layer hidden states of **gpt2-medium** can distinguish truthful from untruthful statements on the repeng_truthful dataset with grouped accuracy above chance.

The comparison across probe methods (dimensionality-based, LAT, logistic regression, PCA-guided) reveals which linear directions in activation space best capture the truth signal. If logistic regression performs best, it suggests the signal benefits from supervised fitting; if unsupervised methods (LAT, PCA-g) match it, the truth direction may be a prominent feature of the geometry even without labels.

Key takeaways:
- The hidden states of gpt2-medium contain a **linearly accessible truth signal** that generalizes across unseen question groups.
- The group-level split ensures we are measuring genuine generalization, not surface-level memorization.
- The next steps (Notebook 3) will test whether this probe **transfers** to entirely different datasets, which is the real test of whether the model has a universal notion of truthfulness.